# Credit Default Modelling

This notebook builds and evaluates models for predicting customer default.

In [136]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import log_loss

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

Separate the predictor variables from the default target and remove client_id.

In [118]:
X = train.drop(columns=["default", "client_id"])
y = train["default"]

X_test = test.drop(columns=["client_id"])

Split the labelled data into training and validation sets. Use stratification to keep the default rate similar in both sets.


In [119]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=50
)


Create lists of columns that are categorical and standardise the remaining numerical features.


In [120]:
categorical_cols = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]


In [121]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numeric_cols
        )
    ]
)


Use logistic regression as the first baseline model.


In [122]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])


In [123]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=50
)

cv_scores = -cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="neg_log_loss"
)

baseline_logloss = cv_scores.mean()

print("Fold Log Loss:", cv_scores)
print("Mean CV Log Loss:", baseline_logloss)
print("Standard Deviation:", cv_scores.std())

Fold Log Loss: [0.43757934 0.42918817 0.44323252 0.43680413 0.43591197]
Mean CV Log Loss: 0.43654322437846715
Standard Deviation: 0.004480704353575942


Use XGBoost as a stronger nonlinear model and evaluate it using the same 5-fold cross-validation setup as the logistic regression baseline.

In [124]:
boost_categorical_cols = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

In [125]:
xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            boost_categorical_cols
        )
    ],
    remainder="passthrough"
)

In [126]:
xgb_model = Pipeline([
    ("preprocessor", xgb_preprocessor),
    (
        "classifier",
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=50,
            n_jobs=-1
        )
    )
])

In [127]:
xgb_scores = -cross_val_score(
    xgb_model,
    X,
    y,
    cv=cv,
    scoring="neg_log_loss"
)

xgb_logloss = xgb_scores.mean()

print("Fold Log Loss:", xgb_scores)
print("Mean CV Log Loss:", xgb_logloss)
print("Standard Deviation:", xgb_scores.std())

Fold Log Loss: [0.42739013 0.41644689 0.43619543 0.43129441 0.42460811]
Mean CV Log Loss: 0.4271869957447052
Standard Deviation: 0.006634221762947777


Use CatBoost as another stronger nonlinear model and evaluate it using the same 5-fold cross-validation setup as the previous models.

In [128]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=5,
    loss_function="Logloss",
    cat_features=boost_categorical_cols,
    verbose=0,
    random_seed=50,
    thread_count=-1
)

In [130]:
cat_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):

    X_train_fold = X.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=5,
        loss_function="Logloss",
        verbose=0,
        random_seed=50,
        thread_count=-1
    )

    cat_model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=boost_categorical_cols
    )

    cat_probs = cat_model.predict_proba(X_valid_fold)[:, 1]

    fold_score = log_loss(
        y_valid_fold,
        cat_probs
    )

    cat_scores.append(fold_score)

    print(f"Fold {fold}: {fold_score:.5f}")

Fold 1: 0.42671
Fold 2: 0.41662
Fold 3: 0.43706
Fold 4: 0.42861
Fold 5: 0.42420


In [137]:
cat_scores = np.array(cat_scores)

cat_logloss = cat_scores.mean()

print("\nFold Log Loss:", cat_scores)
print("Mean CV Log Loss:", cat_logloss)
print("Standard Deviation:", cat_scores.std())


Fold Log Loss: [0.4267056  0.4166158  0.43706394 0.42861445 0.42420285]
Mean CV Log Loss: 0.42664053018775033
Standard Deviation: 0.006617939412801522


In [138]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "XGBoost",
        "CatBoost"
    ],
    "Mean CV Log Loss": [
        baseline_logloss,
        xgb_logloss,
        cat_logloss
    ]
})

results

,Model,Mean CV Log Loss
0,Logistic Regression,0.436543
1,XGBoost,0.427187
2,CatBoost,0.426641
